In [422]:
%pip install pandas numpy seaborn matplotlib scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [423]:
import pandas as pd
import numpy as np

In [424]:
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.linear_model import Ridge, Lasso, LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

In [425]:
df = pd.read_csv('final_data.csv')
df

,Idade,Petroleo,Agua,Gas,Gas_Royalties,Tempo_Producao,Grau_API,Densidade_GLP_Gas,PCS
0,4073,10597.4769,4800.2166,56.5923,1355.916,744,33.3,2.02232,40592.3314
1,3702,3245.1189,8032.5534,16.1271,386.394,744,33.3,2.02232,40592.3314
2,3563,0.0000,0.0000,0.0000,0.000,0,33.3,2.02232,40592.3314
3,4128,2116.3271,5439.4737,25.6074,613.536,714,33.3,2.02232,40592.3314
4,3609,1978.3072,5187.5167,16.1911,387.928,712,33.3,2.02232,40592.3314
...,...,...,...,...,...,...,...,...,...
21454,4742,1374.4294,32.1202,0.9751,25.130,741,19.2,2.06677,40529.9208
21455,4722,0.0000,0.0000,0.0000,0.000,0,19.2,2.06677,40529.9208
21456,4609,979.8785,136.3452,1.3825,35.633,736,19.2,2.06677,40529.9208
21457,4639,2460.3767,30.4620,16.0628,413.996,742,19.2,2.06677,40529.9208


In [ ]:
from math import log

In [ ]:
df['Agua_log'] = df['Agua'].apply(lambda x: log(x) if x > 0 else 0)
df['Gas_log'] = df['Gas'].apply(lambda x: log(x) if x > 0 else 0)
df['Idade2'] = df['Idade'].apply(lambda x: x**2)
df

,Idade,Petroleo,Agua,Gas,Gas_Royalties,Tempo_Producao,Grau_API,Densidade_GLP_Gas,PCS,Agua_log,Gas_log,Idade2
0,4073,10597.4769,4800.2166,56.5923,1355.916,744,33.3,2.02232,40592.3314,8.476416,4.035873,16589329
1,3702,3245.1189,8032.5534,16.1271,386.394,744,33.3,2.02232,40592.3314,8.991258,2.780501,13704804
2,3563,0.0000,0.0000,0.0000,0.000,0,33.3,2.02232,40592.3314,0.000000,0.000000,12694969
3,4128,2116.3271,5439.4737,25.6074,613.536,714,33.3,2.02232,40592.3314,8.601438,3.242881,17040384
4,3609,1978.3072,5187.5167,16.1911,387.928,712,33.3,2.02232,40592.3314,8.554010,2.784462,13024881
...,...,...,...,...,...,...,...,...,...,...,...,...
21454,4742,1374.4294,32.1202,0.9751,25.130,741,19.2,2.06677,40529.9208,3.469485,-0.025215,22486564
21455,4722,0.0000,0.0000,0.0000,0.000,0,19.2,2.06677,40529.9208,0.000000,0.000000,22297284
21456,4609,979.8785,136.3452,1.3825,35.633,736,19.2,2.06677,40529.9208,4.915190,0.323893,21242881
21457,4639,2460.3767,30.4620,16.0628,413.996,742,19.2,2.06677,40529.9208,3.416480,2.776506,21520321


In [429]:
target_column = 'Petroleo'
ignore_columns = [
    #'Gas',
    # 'Agua',
    # 'Idade'
]

In [430]:
sample_df = df.sample(frac=1, random_state=42)
sample_df.drop_duplicates(inplace=True)
for col in sample_df.columns:
    Q1 = sample_df[col].quantile(0.25)
    Q3 = sample_df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    sample_df = sample_df[(sample_df[col] >= lower_bound) & (sample_df[col] <= upper_bound)]

sample_df

,Idade,Petroleo,Agua,Gas,Gas_Royalties,Tempo_Producao,Grau_API,Densidade_GLP_Gas,PCS,Agua_log,Gas_log,Idade2
5527,5636,508.8115,5839.3422,13.8773,414.564,259,22.20,2.03282,39249.2117,8.672373,2.630254,31764496
15902,4453,102.7398,5.0067,3.7128,54.700,676,30.20,2.00144,38798.0539,1.610777,1.311786,19829209
12124,488,7.6646,0.3782,0.2990,4.247,62,30.20,2.00144,38798.0539,-0.972332,-1.207312,238144
1297,6083,125.6144,564.0495,0.2412,7.476,727,30.23,2.12184,36715.5245,6.335142,-1.422129,37002889
2554,3002,69.0662,0.0276,3.0274,22.671,490,28.70,2.04092,36555.2570,-3.589940,1.107704,9012004
...,...,...,...,...,...,...,...,...,...,...,...,...
6265,5463,0.0000,0.0000,0.0000,0.000,0,18.10,2.00677,39989.7428,0.000000,0.000000,29844369
11964,488,102.3119,5.0487,3.9909,56.696,62,30.20,2.00144,38798.0539,1.619131,1.384017,238144
5390,348,578.2746,0.5789,9.2298,264.040,61,24.70,2.01939,40444.9634,-0.546626,2.222437,121104
860,2638,132.0420,3.2312,5.2944,76.693,619,30.20,2.00144,38798.0539,1.172854,1.666650,6959044


In [431]:
X = sample_df.drop(ignore_columns + [target_column], axis=1)
y = sample_df[target_column]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.2, random_state=42)

In [432]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# LinearRegression

In [433]:
model = LinearRegression()
model.fit(X_train_scaled, y_train)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [434]:
print("Regressão Linear")
print(f"RMSE: {np.sqrt(mean_squared_error(y_train, model.predict(X_train_scaled))):.4f} (Treino), {np.sqrt(mean_squared_error(y_test, model.predict(X_test_scaled))):.4f} (Teste)")
print(f"MAE: {mean_absolute_error(y_train, model.predict(X_train_scaled)):.4f} (Treino), {mean_absolute_error(y_test, model.predict(X_test_scaled)):.4f} (Teste)")
print(f"R2: {r2_score(y_train, model.predict(X_train_scaled)):.4f} (Treino), {r2_score(y_test, model.predict(X_test_scaled)):.4f} (Teste)")

Regressão Linear
RMSE: 833.7957 (Treino), 789.1246 (Teste)
MAE: 510.3713 (Treino), 479.7244 (Teste)
R2: 0.5856 (Treino), 0.5720 (Teste)


# Ridge

In [435]:
alphas = np.logspace(-4, 2, 100)

scoring_metric = 'r2'
ridge_model = Ridge(random_state=42)
grid_search_ridge = GridSearchCV(
    estimator=ridge_model,
    param_grid={'alpha': alphas},
    scoring=scoring_metric,
    cv=5,
    verbose=0,
    n_jobs=-1
)

# 2.2 Treinar o GridSearch
grid_search_ridge.fit(X_train_scaled, y_train)

# Obter o melhor modelo e alpha
best_alpha_ridge = grid_search_ridge.best_params_['alpha']
best_ridge_model = grid_search_ridge.best_estimator_

In [436]:
print("Ridge")
print(f"Best alpha: {best_alpha_ridge}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_train, best_ridge_model.predict(X_train_scaled))):.4f} (Treino), {np.sqrt(mean_squared_error(y_test, best_ridge_model.predict(X_test_scaled))):.4f} (Teste)")
print(f"MAE: {mean_absolute_error(y_train, best_ridge_model.predict(X_train_scaled)):.4f} (Treino), {mean_absolute_error(y_test, best_ridge_model.predict(X_test_scaled)):.4f} (Teste)")
print(f"R2: {r2_score(y_train, best_ridge_model.predict(X_train_scaled)):.4f} (Treino), {r2_score(y_test, best_ridge_model.predict(X_test_scaled)):.4f} (Teste)")

Ridge
Best alpha: 5.336699231206313
RMSE: 833.8015 (Treino), 788.9564 (Teste)
MAE: 510.1656 (Treino), 479.5173 (Teste)
R2: 0.5856 (Treino), 0.5722 (Teste)


# Lasso

In [437]:
lasso_model = Lasso(random_state=42, max_iter=2000) 
grid_search_lasso = GridSearchCV(
    estimator=lasso_model,
    param_grid={'alpha': alphas},
    scoring=scoring_metric,
    cv=5,
    verbose=0,
    n_jobs=-1
)

grid_search_lasso.fit(X_train_scaled, y_train)

best_alpha_lasso = grid_search_lasso.best_params_['alpha']
best_lasso_model = grid_search_lasso.best_estimator_

In [438]:
print("Ridge")
print(f"RMSE: {np.sqrt(mean_squared_error(y_train, best_ridge_model.predict(X_train_scaled))):.4f} (Treino), {np.sqrt(mean_squared_error(y_test, best_ridge_model.predict(X_test_scaled))):.4f} (Teste)")
print(f"MAE: {mean_absolute_error(y_train, best_ridge_model.predict(X_train_scaled)):.4f} (Treino), {mean_absolute_error(y_test, best_ridge_model.predict(X_test_scaled)):.4f} (Teste)")
print(f"R2: {r2_score(y_train, best_ridge_model.predict(X_train_scaled)):.4f} (Treino), {r2_score(y_test, best_ridge_model.predict(X_test_scaled)):.4f} (Teste)")

Ridge
RMSE: 833.8015 (Treino), 788.9564 (Teste)
MAE: 510.1656 (Treino), 479.5173 (Teste)
R2: 0.5856 (Treino), 0.5722 (Teste)
